In [1]:
import os
import numpy as np
import soundfile as sf
import librosa
import noisereduce as nr
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib
from imblearn.over_sampling import SMOTE

/home/habib/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


np array from librosa.load, every sr ms get the 13 mfcc using librosa.feature.mfcc, get the mean and std across all the columns in a single row using np.mean and np.std where axis=1 stands for columns, get the duration of the cry, get the energy or average "loadness" from the cry (rms) or root mean square energy. f0 is the fundamental or lowest frequency, zcr or (zero-crossing rate) measures how "piercing" a cry is by measuring how often the sound wave transitions from positive energy to negative energy through the 0-line axis.

In [2]:
# ==========================================================
# 1. EXPANDED FEATURE EXTRACTION (MFCC + DELTAS + SPECTRAL)
# ==========================================================
def clean_and_extract_features(y, sr):
    """
    Isolates local noise profile windows, applies spectral filters, 
    and computes dynamic 2D frequency deltas.
    """
    # Dynamic profile extraction from the initial 500ms block
    num_noise_samples = int(sr * 0.5)
    noise_profile = y[:num_noise_samples] if len(y) > num_noise_samples else y
    cleaned_y = nr.reduce_noise(y=y, sr=sr, y_noise=noise_profile, prop_decrease=0.85)
    
    # 1. Broaden coefficient resolution to 20 filters
    mfccs = librosa.feature.mfcc(y=cleaned_y, sr=sr, n_mfcc=20)
    
    # 2. Extract speed/acceleration deltas to highlight urgency profiles
    mfcc_delta = librosa.feature.delta(mfccs)
    mfcc_delta2 = librosa.feature.delta(mfccs, order=2)
    
    # 3. Capture harmonic texture variations via Spectral Contrast
    spectral_contrast = librosa.feature.spectral_contrast(y=cleaned_y, sr=sr)
    
    # Compress matrix allocations into a comprehensive global 1D vector map
    feature_vector = np.hstack((
        np.mean(mfccs, axis=1), np.std(mfccs, axis=1),
        np.mean(mfcc_delta, axis=1), np.std(mfcc_delta, axis=1),
        np.mean(mfcc_delta2, axis=1), np.std(mfcc_delta2, axis=1),
        np.mean(spectral_contrast, axis=1), np.std(spectral_contrast, axis=1)
    ))
    return feature_vector

# ==========================================================
# 2. MATCHED DATASET LOADER (YOUR CURRENT DIRECTORIES)
# ==========================================================
def load_dataset_paths(dataset_path):
    """
    Crawls your cleaned dataset structure mapping exactly to the 
    discomfort, hungry, and tired directories shown in your file explorer.
    """
    file_paths = []
    labels = []
    
    target_categories = ['hungry', 'tired', 'discomfort']
    
    for category in target_categories:
        folder_dir = os.path.join(dataset_path, category)
        if not os.path.exists(folder_dir):
            print(f"[ERROR] Directory not found: {folder_dir}")
            continue
            
        print(f" -> Indexing files from directory: '{category}'")
        for file_name in os.listdir(folder_dir):
            if file_name.endswith('.wav'):
                file_paths.append(os.path.join(folder_dir, file_name))
                labels.append(category)
                
    return np.array(file_paths), np.array(labels)


In [3]:
# ==========================================================
# 3. MAIN TRAINING RUN
# ==========================================================
if __name__ == "__main__":
    # Your verified path mapping configuration
    DATASET_DIR = "/home/habib/mindcloud/project/dataset"  
    MODEL_OUTPUT = "infant_cry_model.joblib"
    
    print("[1/5] Executing directory crawling across your folders...")
    X_paths, y_labels = load_dataset_paths(DATASET_DIR)
    
    unique, counts = np.unique(y_labels, return_counts=True)
    print(f"Detected Folder Distribution: {dict(zip(unique, counts))}")
    
    # Guard Splitting Integrity: Split 80/20 before running SMOTE synthesis loops
    X_train_paths, X_test_paths, y_train_raw, y_test_raw = train_test_split(
        X_paths, y_labels, test_size=0.20, random_state=42, stratify=y_labels
    )
    
    print("\n[2/5] Cleaning and extracting static validation test features...")
    X_test = []
    for path in X_test_paths:
        y, sr = sf.read(path)
        X_test.append(clean_and_extract_features(y, sr))
    X_test = np.array(X_test)
    y_test = y_test_raw

    print("[3/5] Processing baseline training audio arrays...")
    X_train_raw = []
    for path in X_train_paths:
        y, sr = sf.read(path)
        X_train_raw.append(clean_and_extract_features(y, sr))
    X_train_raw = np.array(X_train_raw)

    print("\n[4/5] INCLUDED STEP: Executing SMOTE oversampling algorithms...")
    # k_neighbors=2 securely handles your small 'tired' training chunk without rendering matrix errors
    smote = SMOTE(random_state=42, k_neighbors=2)
    X_train_balanced, y_train_balanced = smote.fit_resample(X_train_raw, y_train_raw)
    
    balanced_unique, balanced_counts = np.unique(y_train_balanced, return_counts=True)
    print(f"Forcefully Balanced Training Matrix Layout: {dict(zip(balanced_unique, balanced_counts))}")

    print("\n[5/5] Training regularization-bounded Random Forest classifier...")
    # Tree boundary settings to prevent over-memorization of hungry vectors
    clf = RandomForestClassifier(
        n_estimators=250,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features='sqrt',
        random_state=42
    )
    clf.fit(X_train_balanced, y_train_balanced)
    
    # Generate finalized evaluation reports
    y_pred = clf.predict(X_test)
    
    print("\n================ UPDATED CONFUSION MATRIX ================")
    print(confusion_matrix(y_test, y_pred, labels=['hungry', 'tired', 'discomfort']))
    
    print("\n============= UPDATED CLASSIFICATION REPORT =============")
    print(classification_report(y_test, y_pred))
    
    # Save the pipeline using joblib optimized mappings
    joblib.dump(clf, MODEL_OUTPUT)
    print(f"\n[SUCCESS] Serialized model successfully saved to: {MODEL_OUTPUT}")

[1/5] Executing directory crawling across your folders...
 -> Indexing files from directory: 'hungry'
 -> Indexing files from directory: 'tired'
 -> Indexing files from directory: 'discomfort'
Detected Folder Distribution: {np.str_('discomfort'): np.int64(51), np.str_('hungry'): np.int64(382), np.str_('tired'): np.int64(24)}

[2/5] Cleaning and extracting static validation test features...


ParameterError: Frequency band exceeds Nyquist. Reduce either fmin or n_bands.